In [1]:
import nsepy

In [3]:
from nsepy import get_history
from nsepy.derivatives import get_expiry_date
from datetime import date
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [4]:
winner_portfolio = pd.read_csv("winner_portfolio.csv")
loser_portfolio = pd.read_csv("loser_portfolio.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'winner_portfolio.csv'

In [135]:
start_date = pd.to_datetime(winner_portfolio['Date'][0]).date()   #getting start date from winner portfolio
end_date = start_date + pd.Timedelta(days=1)  #end date is start date + 1 day

In [136]:
expiry_date = list(get_expiry_date(year=start_date.year,month=start_date.month+1)) #getting expiry date of next month
expiry_date.sort() #sorting expiry date
for i in range(1,len(expiry_date)):
    if expiry_date[-i].weekday() != 1: # removing expiry date which is Tuesday
        expiry_date = expiry_date[-i]
        break

In [137]:
current_price = pd.DataFrame(columns=['Date','Symbol','Close','CE/PE']) #creating dataframe to store current price of stocks

In [138]:
# get last row first three columns values
winner_symbols = winner_portfolio.iloc[0,1:4].tolist()
for symbol in winner_symbols:
    current_price = current_price.append({'Date':start_date,'Symbol':symbol,'Close':get_history(symbol=symbol,start=start_date,end=end_date)['Close'][0],'CE/PE':'CE'},ignore_index=True)

looser_symbols = loser_portfolio.iloc[0,1:4].tolist()
for symbol in looser_symbols:
    current_price = current_price.append({'Date':start_date,'Symbol':symbol,'Close':get_history(symbol=symbol,start=start_date,end=end_date)['Close'][0],'CE/PE':'PE'},ignore_index=True)


In [139]:
current_price

,Date,Symbol,Close,CE/PE
0,2023-02-24,SIEMENS,3246.50,CE
1,2023-02-24,ITC,385.10,CE
2,2023-02-24,GAIL,103.45,CE
3,2023-02-24,ADANIGREEN,486.50,PE
4,2023-02-24,ADANITRANS,712.30,PE
5,2023-02-24,ADANIENT,1315.65,PE


In [140]:
current_option = pd.DataFrame(columns=['Date','Symbol','ExpiryDate','StockClose','StrikePrice','CE/PE','OptionClose']) #creating dataframe to store current option price of stocks

for stock in range(len(current_price)): #looping through each stock
    StrikePrice = current_price['Close'][stock] - (current_price['Close'][stock] % 5) #getting strike price
    for i in range(1,10): #looping through 10 strike prices
        stock_opt = get_history(symbol=current_price['Symbol'][stock],
                                start=start_date,
                                end=end_date,
                                option_type=current_price['CE/PE'][stock],
                                strike_price=StrikePrice + (i*5), #adding 5 to strike price
                                expiry_date=expiry_date)

        if len(stock_opt) != 0:
            current_option = current_option.append({'Date':start_date,'Symbol':current_price['Symbol'][stock],'ExpiryDate':expiry_date,'StockClose':current_price['Close'][stock],'StrikePrice':StrikePrice,'CE/PE':current_price['CE/PE'][stock],'OptionClose':stock_opt['Close'][0]},ignore_index=True) #appending option price to dataframe
            break


In [141]:
current_option

,Date,Symbol,ExpiryDate,StockClose,StrikePrice,CE/PE,OptionClose
0,2023-02-24,SIEMENS,2023-03-29,3246.50,3245.0,CE,98.7
1,2023-02-24,ITC,2023-03-29,385.10,385.0,CE,6.3
2,2023-02-24,GAIL,2023-03-29,103.45,100.0,CE,3.0
3,2023-02-24,ADANIENT,2023-03-29,1315.65,1315.0,PE,170.0


In [142]:
# there are some stocks which don't have option chain

In [143]:
current_option.to_csv(f"{current_option['Date'][0]}_option.csv",index=False)